In [1]:
# =============================================================================
# Korea Trade Data Collector (V7)  -- 전국 단위 품목별 수출입
# V7 변경 (2026-10-17)
# - HS 코드 목록: hs_code_map_v3.build_collection_list()
#     정적 목록(HS_CODE_MAP) + DB 에서 실제로 쓰는 코드(fd_driver_map_2d, kr_company_hs_map,
#     korea_company_region_hscode_map)를 자동으로 합친다. 새 드라이버 코드가 생겨도 목록 파일을 고칠 필요 없음
# - 4자리 코드는 4자리로 직접 요청 (하위 6자리 합으로 대신하지 않음)
# - 끊긴 코드 자동 점검: 코드별 최신월이 전체 최신월보다 STALE_MONTHS 넘게 뒤처지면 경고 + 그 지점부터 자동 재수집
#   처음 보는 코드(DB에 없음)는 NEW_CODE_START 부터 과거까지 자동 수집
# - 코드별 시작월은 분기 첫 달로 맞춤 (분기 합계가 일부 달로 덮어써지는 것 방지)
# - 10자리: ALLOW_10DIGIT=True 일 때만 10자리 그대로 요청. 먼저 RUN_PROBE_10DIGIT 로 API 지원 여부 확인
# - 수집 후 커버리지 리포트(CSV): 코드별 최신월, 상태(ok/stale/new/empty)
#
# V6 변경 (2026-10-01)
# - HS 코드 목록: hs_code_map_v2 (누락 코드 6개 추가: 030341~030344, 0207, 190490)
# - 서비스키/DB 비밀번호를 코드에서 제거 → DATA/KEYS.py, DATA/config.py 에서 읽음
# - TARGET_CODES: 특정 코드만 골라 과거부터 재수집 가능 (누락 코드 백필용)
# - YoY 계산용으로 시작월 12개월 전부터 함께 받음 (V5는 1년치만 받아 YoY가 비어 있었음)
# - DB 저장: 기존 키는 건너뛰던 방식 → 값 갱신(UPSERT). 수정치 반영, 전체 키 조회 제거로 빨라짐
# - AUTO_UPLOAD: 업로드 여부를 설정으로 지정 가능 (None 이면 기존처럼 물어봄)
# - HS별 1회 작업(내부에서 12개월 window로 자동 분할)
# - YYYYMM 범위 수집 (START_YYYYMM ~ END_YYYYMM)
# - API 제약: 조회기간 1년(12개월) 이내 → 자동 분할 후 concat
# - 안전 파싱 / 재시도 / 세션 풀 / 6자리 HS 요청 대응
# - 월/분기 집계 + YoY + Long format + DB 업로드
# =============================================================================

import re
import time
import logging
import warnings

import requests
import xmltodict
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd
from tqdm import tqdm

from sqlalchemy import create_engine, text
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter

# -----------------------------------------------------------------------------
# 경로 설정 (노트북 / 데스크탑 공용)
#   1순위: 현재 위치에서 상위 폴더로 올라가며 DATA 폴더 자동 탐색
#   2순위: 두 PC의 알려진 고정 경로 fallback
#   → "from DATA.xxx import ..." 가 작동하려면 DATA의 "상위 폴더(프로젝트 루트)"가
#     sys.path에 있어야 하므로, 루트와 DATA 폴더를 모두 추가한다.
# -----------------------------------------------------------------------------
import sys
import os
from pathlib import Path

# 두 PC의 알려진 DATA 폴더 경로 (fallback용)
KNOWN_DATA_PATHS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA",          # 데스크탑
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA",  # 노트북
]

def setup_universal_paths():
    """어떤 PC에서도 작동하는 범용 경로 설정: DATA 폴더 자동 탐색 + 고정 경로 fallback"""
    # 1) 현재 위치 기준 상위 폴더 자동 탐색
    current = Path.cwd()
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print(f"[경로 설정] 자동 탐색 성공 | 프로젝트 루트: {parent}")
            print(f"[경로 설정] DATA 폴더: {data_folder}")
            return str(data_folder)

    # 2) 고정 경로 fallback (자동 탐색 실패 시)
    for dp in KNOWN_DATA_PATHS:
        if os.path.exists(dp):
            root = str(Path(dp).parent)
            for p in (root, dp):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print(f"[경로 설정] 고정 경로 사용 | 프로젝트 루트: {root}")
            print(f"[경로 설정] DATA 폴더: {dp}")
            return dp

    raise FileNotFoundError(
        f"DATA 폴더를 찾을 수 없습니다.\n"
        f"현재 위치: {current}\n"
        f"상위 폴더에 DATA 폴더가 있는지, 또는 KNOWN_DATA_PATHS의 경로가 맞는지 확인하세요."
    )

DATA_PATH = setup_universal_paths()

from DATA.hs_code_map_v3 import HS_CODE_MAP, validate_hs_code_map, build_collection_list

_bad = validate_hs_code_map(HS_CODE_MAP)
if _bad:
    raise ValueError(f"hs_code_map_v3 에 형식이 잘못된 코드가 있습니다 (문자열/짝수 자릿수 아님): {_bad}")
# -----------------------------------------------------------------------------
# 로깅 설정
# -----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# -----------------------------------------------------------------------------
# 설정
# -----------------------------------------------------------------------------
# 서비스키: DATA/KEYS.py 의 KEYS 에서 아래 이름 순서로 찾음 (공공데이터포털 키)
SERVICE_KEY_NAMES = ("CUSTOMS", "ODPD")

START_YYYYMM = "202601"
END_YYYYMM   = "202612"
REGION_NAME  = "전국"

TARGET_CODES = None
#
# # YoY 계산용으로 시작월 12개월 전부터 함께 받기 (권장 True)
YOY_LOOKBACK = True
#
# # DB 업로드: None = 실행 중 물어봄 / True = 자동 업로드 / False = 업로드 안 함
AUTO_UPLOAD = None

# ---- V7: 수집 목록 / 끊김 점검 / 10자리 ----
USE_DEMAND_CODES   = True      # DB 수요 코드(드라이버·기업 매핑)를 목록에 자동 추가
STALE_MONTHS       = 2         # 전체 최신월보다 이만큼 넘게 뒤처진 코드는 '끊김'으로 보고 재수집
AUTO_BACKFILL      = True      # 끊긴 코드·처음 보는 코드를 자동으로 과거부터 채움
NEW_CODE_START     = "201001"  # 처음 보는 코드의 수집 시작월
ALLOW_10DIGIT      = False     # True: 10자리 코드를 10자리로 요청 (RUN_PROBE_10DIGIT 로 지원 확인 후)
RUN_PROBE_10DIGIT  = False     # True: 수집 전에 10자리 지원 여부만 시험하고 끝냄
PROBE_HS6          = "854232"  # 10자리 시험에 쓸 6자리 코드
MONTHLY_TABLE      = "korea_monthly_trade_data"
QUARTERLY_TABLE    = "korea_quarterly_trade_data"

# TARGET_CODES = ["030341", "030342", "030343", "030344", "030354", "030389",
#                "030487", "030633", "0207", "190490"]
# START_YYYYMM = "200701"
# AUTO_UPLOAD = True

RETRIES = 3
TIMEOUT = 60

MAX_WORKERS   = 8        # 8~10 추천
REQUEST_DELAY = 0.10     # 서버 안정성 위해 0.1~0.2 추천
CHUNK_SIZE    = 1000

# DB 접속: DATA/config.py 사용 (비밀번호를 코드에 적지 않음)
import config as investar_config
ENGINE = investar_config.get_engine(investar_config.get_db_info())

# 서비스키 로드
from urllib.parse import quote
try:
    from DATA.KEYS import KEYS
except Exception:
    KEYS = {}
_raw_key = next((KEYS.get(k) for k in SERVICE_KEY_NAMES if KEYS.get(k)), None) or os.environ.get("CUSTOMS_SERVICE_KEY")
if not _raw_key:
    raise RuntimeError(f"서비스키가 없습니다. DATA/KEYS.py 에 KEYS['CUSTOMS'] = '...' 를 추가하세요. "
                       f"(현재 KEYS 이름: {sorted(KEYS)})")
# URL에 직접 붙이므로 인코딩된 형태로 맞춤 (이미 %가 있으면 인코딩된 키로 간주)
SERVICE_KEY = _raw_key if "%" in _raw_key else quote(_raw_key, safe="")


def mask_key(msg) -> str:
    out = str(msg)
    for k in {SERVICE_KEY, _raw_key}:
        if k:
            out = out.replace(k, "***")
    return out

# Unique HS Code 리스트 (정적 목록). 실제 수집 목록은 main() 에서 build_collection_list() 로 만든다
hs_code_list: list = list(HS_CODE_MAP.keys())

# -----------------------------------------------------------------------------
# 유틸
# -----------------------------------------------------------------------------
def norm_yymm(yymm: str) -> str:
    """YYYYMM 6자리 검증"""
    yymm = str(yymm).strip()
    if not re.fullmatch(r"\d{6}", yymm):
        raise ValueError(f"Invalid YYYYMM (must be 6 digits): {yymm}")
    mm = int(yymm[4:6])
    if not (1 <= mm <= 12):
        raise ValueError(f"Invalid month in YYYYMM: {yymm}")
    return yymm

def yymm_to_int(yymm: str) -> int:
    """YYYYMM -> month index (연*12 + (월-1))"""
    yymm = norm_yymm(yymm)
    return int(yymm[:4]) * 12 + (int(yymm[4:6]) - 1)

def int_to_yymm(m: int) -> str:
    """month index -> YYYYMM"""
    yyyy = m // 12
    mm = (m % 12) + 1
    return f"{yyyy:04d}{mm:02d}"

def split_into_12m_windows(start_yymm: str, end_yymm: str):
    """
    [start_yymm, end_yymm] (포함) 범위를
    API 제약(<=12개월)에 맞게 구간 리스트로 분할.
    """
    s = yymm_to_int(start_yymm)
    e = yymm_to_int(end_yymm)
    if e < s:
        raise ValueError("end_yymm must be >= start_yymm")

    windows = []
    cur = s
    while cur <= e:
        win_end = min(cur + 11, e)  # 최대 12개월
        windows.append((int_to_yymm(cur), int_to_yymm(win_end)))
        cur = win_end + 1
    return windows

def parse_items_to_df(response_text: str) -> pd.DataFrame:
    """
    API 응답(XML)을 안전하게 파싱해 items/item을 DF로 반환.
    정상 구조가 아니면 빈 DF.
    """
    try:
        parsed = xmltodict.parse(response_text)
    except Exception:
        return pd.DataFrame()

    if not isinstance(parsed, dict):
        return pd.DataFrame()

    resp = parsed.get("response")
    if not isinstance(resp, dict):
        return pd.DataFrame()

    body = resp.get("body")
    if not isinstance(body, dict):
        return pd.DataFrame()

    items = body.get("items")
    if not isinstance(items, dict):
        return pd.DataFrame()

    item = items.get("item")
    if item is None:
        return pd.DataFrame()

    if isinstance(item, dict):
        item = [item]
    if not isinstance(item, list):
        return pd.DataFrame()

    return pd.DataFrame(item)

def extract_api_header(response_text: str):
    """resultCode/resultMsg 추출 (없으면 None,None)"""
    try:
        parsed = xmltodict.parse(response_text)
        if not isinstance(parsed, dict):
            return None, None
        resp = parsed.get("response")
        if not isinstance(resp, dict):
            return None, None
        header = resp.get("header")
        if not isinstance(header, dict):
            return None, None
        return header.get("resultCode"), header.get("resultMsg")
    except Exception:
        return None, None

def make_session(max_workers: int) -> requests.Session:
    """연결 재사용 + pool 확장"""
    sess = requests.Session()
    adapter = HTTPAdapter(pool_connections=max_workers, pool_maxsize=max_workers)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    return sess

# -----------------------------------------------------------------------------
# 수집 함수 (단일 window)
# -----------------------------------------------------------------------------
def get_country_export_by_item(session: requests.Session,
                               service_key: str,
                               start_yymm: str,
                               end_yymm: str,
                               hs_code_raw) -> pd.DataFrame:
    """
    단일 window(start_yymm~end_yymm) 수집
    - 요청 HS: 4/6자리 그대로, 10자리는 ALLOW_10DIGIT 일 때만 그대로 (아니면 6자리로 절단)
    - 원본 root_hs_code는 그대로 저장
    """
    start_yymm = norm_yymm(start_yymm)
    end_yymm   = norm_yymm(end_yymm)

    hs_code_raw = str(hs_code_raw).strip()
    # 4/6자리는 그대로, 10자리는 ALLOW_10DIGIT 일 때만 10자리 (아니면 6자리로 절단, V6 방식)
    hs_code_req = hs_code_raw if (len(hs_code_raw) <= 6 or ALLOW_10DIGIT) else hs_code_raw[:6]

    url = (
        "https://apis.data.go.kr/1220000/Itemtrade/getItemtradeList"
        f"?serviceKey={service_key}"
        f"&strtYymm={start_yymm}"
        f"&endYymm={end_yymm}"
        f"&hsSgn={hs_code_req}"
    )

    for attempt in range(RETRIES):
        try:
            r = session.get(url, timeout=TIMEOUT)

            # 재시도 상태코드
            if r.status_code in (429, 502, 503, 504):
                if attempt < RETRIES - 1:
                    backoff = 2 ** attempt
                    logger.warning(f"Retryable {r.status_code} HS {hs_code_raw} ({start_yymm}-{end_yymm}) sleep {backoff}s")
                    time.sleep(backoff)
                    continue
                else:
                    logger.error(f"Retryable {r.status_code} but retries exhausted HS {hs_code_raw} ({start_yymm}-{end_yymm})")
                    return pd.DataFrame()

            r.raise_for_status()

            df = parse_items_to_df(r.text)
            if df.empty:
                code, msg = extract_api_header(r.text)
                if code or msg:
                    logger.warning(f"No/Err data HS {hs_code_raw} (req={hs_code_req}) {start_yymm}-{end_yymm} code={code} msg={msg}")
                else:
                    logger.warning(f"No data HS {hs_code_raw} (req={hs_code_req}) {start_yymm}-{end_yymm} head={r.text[:200]}")
                return pd.DataFrame()

            df["root_hs_code"] = hs_code_raw
            df["hsSgn_req"] = hs_code_req
            df["req_start_yymm"] = start_yymm
            df["req_end_yymm"] = end_yymm

            if REQUEST_DELAY > 0:
                time.sleep(REQUEST_DELAY)

            return df

        except requests.exceptions.RequestException as e:
            if attempt < RETRIES - 1:
                backoff = 2 ** attempt
                logger.warning(f"Attempt {attempt+1}/{RETRIES} failed HS {hs_code_raw} ({start_yymm}-{end_yymm}): {mask_key(e)} | sleep {backoff}s")
                time.sleep(backoff)
                continue
            logger.error(f"All retries failed HS {hs_code_raw} ({start_yymm}-{end_yymm}): {mask_key(e)}")
            return pd.DataFrame()

        except Exception as e:
            logger.error(f"Unexpected error HS {hs_code_raw} ({start_yymm}-{end_yymm}): {mask_key(e)}")
            return pd.DataFrame()

# -----------------------------------------------------------------------------
# 수집 함수 (HS당 여러 window 자동 분할)
# -----------------------------------------------------------------------------
def fetch_hs_multiwindow(session: requests.Session,
                         service_key: str,
                         start_yymm: str,
                         end_yymm: str,
                         hs_code_raw) -> pd.DataFrame:
    """
    HS 하나에 대해 전체 기간을 12개월 window로 쪼개서 수집 후 concat
    """
    windows = split_into_12m_windows(start_yymm, end_yymm)
    dfs = []

    for (s, e) in windows:
        df = get_country_export_by_item(session, service_key, s, e, hs_code_raw)
        if df is not None and not df.empty:
            dfs.append(df)

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()

# -----------------------------------------------------------------------------
# 전처리/집계
# -----------------------------------------------------------------------------
def process_and_aggregate(df: pd.DataFrame, region_name: str):
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    # 총계 제거(있을 때만)
    if "year" in df.columns:
        df = df[df["year"] != "총계"].copy()

    # 날짜 처리
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if "year" in df.columns:
            df["new_date"] = pd.to_datetime(df["year"].astype(str).str.replace(".", "-"), errors="coerce") + MonthEnd(0)
        else:
            df["new_date"] = pd.NaT

    df = df.dropna(subset=["new_date"]).copy()
    df = df.set_index("new_date")

    df["new_year"] = df.index.year
    df["new_quarter"] = df.index.quarter
    df["new_month"] = df.index.month

    numeric_cols = ["balPayments", "expDlr", "expWgt", "impDlr", "impWgt"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
        else:
            df[col] = 0.0

    base_cols = ["hsCode", "statKor", "root_hs_code", "hsSgn_req"]
    time_cols = ["new_year", "new_quarter", "new_month"]
    keep_cols = [c for c in (base_cols + time_cols + numeric_cols) if c in df.columns]

    clean_df = df[keep_cols].copy()
    clean_df["region"] = region_name

    agg_dict = {"balPayments": "sum", "expDlr": "sum", "impDlr": "sum", "expWgt": "sum", "impWgt": "sum"}

    by_m = (clean_df.groupby(["root_hs_code", "new_year", "new_quarter", "new_month"])
            .agg(agg_dict)
            .reset_index())
    by_m["region"] = region_name

    by_q = (clean_df.groupby(["root_hs_code", "new_year", "new_quarter"])
            .agg(agg_dict)
            .reset_index())
    by_q["region"] = region_name

    return by_q, by_m

def add_yoy_growth(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    if df.empty:
        return df

    out = df.copy()

    if steps == 12:
        out["date"] = pd.to_datetime(out["new_year"].astype(str) + "-" +
                                     out["new_month"].astype(str).str.zfill(2) + "-01") + MonthEnd(0)
    elif steps == 4:
        end_month_map = {1: "03", 2: "06", 3: "09", 4: "12"}
        end_month = out["new_quarter"].map(end_month_map)
        out["date"] = pd.to_datetime(out["new_year"].astype(str) + "-" + end_month + "-01") + MonthEnd(0)
    else:
        raise ValueError("steps는 12(월) 또는 4(분기)만 지원합니다.")

    out = out.sort_values(["root_hs_code", "date"])
    g = out.groupby("root_hs_code")
    out["expDlr_yoy"] = g["expDlr"].transform(lambda x: x.pct_change(periods=steps))
    out["impDlr_yoy"] = g["impDlr"].transform(lambda x: x.pct_change(periods=steps))

    return out

def reshape_to_long(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    id_vars = ["date", "root_hs_code"]
    value_vars = ["balPayments", "expDlr", "impDlr", "expWgt", "impWgt", "expDlr_yoy", "impDlr_yoy"]
    value_vars = [c for c in value_vars if c in df.columns]

    long_df = df.melt(id_vars=id_vars, value_vars=value_vars,
                      var_name="indicator", value_name="value")
    long_df = long_df.dropna(subset=["value"])
    return long_df

# -----------------------------------------------------------------------------
# DB 업로드
# -----------------------------------------------------------------------------
def upload_to_db_optimized(df_long: pd.DataFrame, table_name: str):
    """INSERT ... ON DUPLICATE KEY UPDATE 로 저장 (새 키는 추가, 기존 키는 값 갱신)."""
    if df_long.empty:
        logger.info(f"업로드할 데이터가 비어있습니다 ({table_name}).")
        return

    df_long = df_long.copy()
    df_long["date"] = pd.to_datetime(df_long["date"]).dt.date
    df_long["root_hs_code"] = df_long["root_hs_code"].astype(str)
    df_long = df_long.replace([np.inf, -np.inf], np.nan).dropna(subset=["value"])
    df_long["value"] = df_long["value"].astype(float)
    recs = df_long[["date", "root_hs_code", "indicator", "value"]].to_dict("records")

    create_sql = text(f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
        `date` DATE,
        `root_hs_code` VARCHAR(20),
        `indicator` VARCHAR(50),
        `value` DOUBLE,
        PRIMARY KEY (`date`, `root_hs_code`, `indicator`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)
    upsert_sql = text(f"""
    INSERT INTO `{table_name}` (`date`, `root_hs_code`, `indicator`, `value`)
    VALUES (:date, :root_hs_code, :indicator, :value)
    ON DUPLICATE KEY UPDATE `value` = VALUES(`value`)
    """)

    logger.info(f"업로드 대상({table_name}): {len(recs):,}건 (새 키 추가 + 기존 키 갱신)")
    with ENGINE.begin() as conn:
        conn.execute(create_sql)
        for i in tqdm(range(0, len(recs), CHUNK_SIZE), desc=f"DB 업로드({table_name})"):
            conn.execute(upsert_sql, recs[i:i + CHUNK_SIZE])
    logger.info(f"총 {len(recs):,}건 저장 완료 ({table_name})")

# =============================================================================
# V7: 커버리지 점검 (코드별 최신월) / 10자리 시험
# =============================================================================
def quarter_floor(yymm: str) -> str:
    """YYYYMM -> 그 분기의 첫 달"""
    y, m = int(yymm[:4]), int(yymm[4:6])
    return f"{y:04d}{(m - 1) // 3 * 3 + 1:02d}"


def load_coverage(table: str = MONTHLY_TABLE) -> pd.DataFrame:
    """코드별 최초·최신월 (expDlr 기준)."""
    q = text(f"SELECT root_hs_code AS code, MIN(`date`) AS first_date, MAX(`date`) AS last_date "
             f"FROM `{table}` WHERE indicator = 'expDlr' GROUP BY root_hs_code")
    try:
        with ENGINE.connect() as conn:
            df = pd.read_sql(q, conn)
    except Exception as e:
        logger.warning(f"커버리지 조회 실패 ({table}): {e}")
        return pd.DataFrame(columns=["code", "first_date", "last_date"])
    df["code"] = df["code"].astype(str)
    df["first_date"] = pd.to_datetime(df["first_date"])
    df["last_date"] = pd.to_datetime(df["last_date"])
    return df


def classify_coverage(codes: list, cov: pd.DataFrame) -> pd.DataFrame:
    """코드별 상태: new(DB에 없음) / stale(뒤처짐) / ok. 기준 = 테이블 전체 최신월."""
    c = pd.DataFrame({"code": [str(x) for x in codes]}).merge(cov, on="code", how="left")
    ref = cov["last_date"].max() if len(cov) else pd.NaT     # 테이블 전체 최신월 기준
    def lag(d):
        if pd.isna(d) or pd.isna(ref):
            return np.nan
        return (ref.year - d.year) * 12 + (ref.month - d.month)
    c["lag_months"] = c["last_date"].map(lag)
    c["status"] = np.where(c["last_date"].isna(), "new",
                   np.where(c["lag_months"] > STALE_MONTHS, "stale", "ok"))
    c.attrs["ref_month"] = ref
    return c


def plan_code_starts(cls: pd.DataFrame, regular_start: str) -> dict:
    """코드별 수집 시작월 (분기 첫 달로 맞춤)."""
    starts = {}
    for _, r in cls.iterrows():
        st = regular_start
        if AUTO_BACKFILL and r["status"] == "new":
            st = min(st, NEW_CODE_START)
        elif AUTO_BACKFILL and r["status"] == "stale":
            back = (r["last_date"] - pd.DateOffset(months=2)).strftime("%Y%m")   # 마지막 자료 2개월 전부터 겹쳐 받기
            if YOY_LOOKBACK:
                back = int_to_yymm(yymm_to_int(back) - 12)
            st = min(st, back)
        starts[r["code"]] = quarter_floor(st)
    return starts


def probe_10digit(hs6: str = PROBE_HS6, yymm: str = None) -> None:
    """6자리 요청 응답의 hsCode 자릿수를 보고, 그중 10자리 코드 하나로 다시 요청해 지원 여부를 확인."""
    yymm = yymm or int_to_yymm(yymm_to_int(pd.Timestamp.today().strftime("%Y%m")) - 3)
    sess = make_session(1)
    def call(code):
        url = ("https://apis.data.go.kr/1220000/Itemtrade/getItemtradeList"
               f"?serviceKey={SERVICE_KEY}&strtYymm={yymm}&endYymm={yymm}&hsSgn={code}")
        r = sess.get(url, timeout=TIMEOUT)
        return parse_items_to_df(r.text), extract_api_header(r.text)
    df6, h6 = call(hs6)
    print(f"[10자리 시험] {hs6} ({yymm}) -> 헤더 {h6}, 행 {len(df6)}")
    if df6.empty or "hsCode" not in df6.columns:
        print("  응답에 hsCode 열이 없어 시험할 수 없습니다.")
        sess.close()
        return
    codes = df6["hsCode"].astype(str).str.replace(r"\D", "", regex=True)
    print("  응답 hsCode 자릿수 분포:", codes.str.len().value_counts().to_dict())
    ten = [c for c in codes if len(c) == 10]
    if not ten:
        print("  응답에 10자리 코드가 없습니다 -> 이 API 는 6자리 단위까지만 주는 것으로 보입니다 (ALLOW_10DIGIT=False 유지).")
        sess.close()
        return
    df10, h10 = call(ten[0])
    print(f"  10자리 {ten[0]} 요청 -> 헤더 {h10}, 행 {len(df10)}")
    if len(df10):
        print("  -> 10자리 요청 지원. 필요한 10자리 코드가 있으면 ALLOW_10DIGIT=True 로 수집하세요.")
        display(df10.head())
    else:
        print("  -> 10자리 요청은 자료를 주지 않습니다 (ALLOW_10DIGIT=False 유지).")
    sess.close()


# =============================================================================
# 메인
# =============================================================================
def main():
    t0 = time.time()
    norm_yymm(START_YYYYMM)
    norm_yymm(END_YYYYMM)

    regular_start = START_YYYYMM
    if YOY_LOOKBACK:
        regular_start = int_to_yymm(yymm_to_int(START_YYYYMM) - 12)   # YoY 계산용 12개월 앞부터

    # (1) 수집 목록
    if TARGET_CODES:
        items = [{"code": str(c).strip(), "name": HS_CODE_MAP.get(str(c).strip(), ""), "sources": "target",
                  "in_static": str(c).strip() in HS_CODE_MAP} for c in TARGET_CODES]
        logger.info(f"TARGET_CODES 지정: {[i['code'] for i in items]}")
    else:
        items = build_collection_list(ENGINE, use_demand=USE_DEMAND_CODES, allow_10digit=ALLOW_10DIGIT)
    item_df = pd.DataFrame(items)
    extra = item_df[~item_df["in_static"]]
    logger.info(f"수집 목록 {len(item_df)}개 (정적 {item_df['in_static'].sum()}, DB 수요 추가 {len(extra)})")
    if len(extra):
        logger.info(f"DB 수요로 추가된 코드: {extra['code'].tolist()[:50]}")

    # (2) 커버리지 점검 -> 코드별 시작월
    cov_before = load_coverage()
    cls = classify_coverage(item_df["code"].tolist(), cov_before)
    ref = cls.attrs.get("ref_month")
    n_new, n_stale = (cls.status == "new").sum(), (cls.status == "stale").sum()
    logger.info(f"커버리지 (기준월 {ref:%Y-%m} ): ok {(cls.status=='ok').sum()} | 끊김 {n_stale} | 처음 {n_new}"
                if pd.notna(ref) else f"커버리지: DB 자료 없음 (처음 {n_new})")
    if n_stale:
        st = cls[cls.status == "stale"].sort_values("last_date")
        logger.warning("끊긴 코드 (최신월): " + ", ".join(f"{c}({d:%Y-%m})" for c, d in zip(st.code, st.last_date)))
    starts = plan_code_starts(cls, regular_start)
    HS_CODES = item_df["code"].tolist()
    n_long = sum(1 for c in HS_CODES if starts[c] < quarter_floor(regular_start))
    logger.info(f"수집기간: 기본 {quarter_floor(regular_start)}~{END_YYYYMM} | 과거 채우기 대상 {n_long}개 코드")

    # (3) 수집
    session = make_session(MAX_WORKERS)
    all_dfs, errors, empty_codes = [], [], []
    try:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_map = {executor.submit(fetch_hs_multiwindow, session, SERVICE_KEY, starts[hs], END_YYYYMM, hs): hs
                          for hs in HS_CODES}
            for future in tqdm(as_completed(future_map), total=len(future_map), desc="API 요청"):
                hs = future_map[future]
                try:
                    df = future.result(timeout=600)
                    if df is not None and not df.empty:
                        all_dfs.append(df)
                    else:
                        empty_codes.append(hs)
                except Exception as e:
                    logger.error(f"HS {hs} 처리 실패: {mask_key(e)}")
                    errors.append(hs)
    finally:
        session.close()
    logger.info(f"수집 종료. 소요: {time.time()-t0:.2f}초 | 실패 HS: {len(errors)} | 빈결과 HS: {len(empty_codes)}")
    if not all_dfs:
        logger.error("수집된 데이터가 없습니다.")
        return None

    final_df = pd.concat(all_dfs, ignore_index=True)
    logger.info(f"총 수집 레코드: {len(final_df):,}건")
    processed_q, processed_m = process_and_aggregate(final_df, REGION_NAME)
    logger.info(f"월별 집계: {len(processed_m):,}건 | 분기별 집계: {len(processed_q):,}건")
    if processed_m.empty:
        logger.warning("처리할 월별 데이터가 없습니다.")
        return None

    monthly_with_yoy = add_yoy_growth(processed_m, steps=12)
    quarterly_with_yoy = add_yoy_growth(processed_q, steps=4)
    monthly_long = reshape_to_long(monthly_with_yoy)
    quarterly_long = reshape_to_long(quarterly_with_yoy)
    logger.info(f"월별 long: {len(monthly_long):,}건 | 분기별 long: {len(quarterly_long):,}건")

    if AUTO_UPLOAD is None:
        do_upload = input("데이터베이스에 업로드하시겠습니까? (y/n): ").strip().lower() == "y"
    else:
        do_upload = bool(AUTO_UPLOAD)
    if do_upload:
        upload_to_db_optimized(monthly_long, MONTHLY_TABLE)
        upload_to_db_optimized(quarterly_long, QUARTERLY_TABLE)

    # (4) 수집 후 커버리지 리포트
    cov_after = load_coverage() if do_upload else cov_before
    rep = classify_coverage(HS_CODES, cov_after).merge(
        item_df[["code", "name", "sources", "in_static"]], on="code", how="left")
    rep = rep.merge(cls[["code", "status", "last_date"]].rename(
        columns={"status": "status_before", "last_date": "last_before"}), on="code", how="left")
    rep["api_empty"] = rep["code"].isin(empty_codes)
    rep["note"] = np.where(rep["api_empty"] & rep["status"].isin(["new", "stale"]),
                           "API 빈 응답 - HS 개정·단종 코드일 수 있음 (확인 필요)", "")
    still = rep[rep["status"].isin(["new", "stale"])]
    logger.info(f"수집 후: ok {(rep.status=='ok').sum()} | 여전히 끊김/없음 {len(still)}")
    if len(still):
        display(still[["code", "name", "sources", "status", "last_date", "api_empty", "note"]])
    try:
        out_dir = Path(DATA_PATH).parent / "reports" / "trade_coverage"
        out_dir.mkdir(parents=True, exist_ok=True)
        p = out_dir / f"trade_coverage_{pd.Timestamp.today():%Y%m%d_%H%M}.csv"
        rep.to_csv(p, index=False, encoding="utf-8-sig")
        logger.info(f"커버리지 리포트: {p}")
    except Exception as e:
        logger.warning(f"리포트 저장 실패: {e}")

    logger.info("전체 파이프라인 완료")
    return {"final_raw": final_df, "monthly": monthly_with_yoy, "quarterly": quarterly_with_yoy,
            "monthly_long": monthly_long, "quarterly_long": quarterly_long,
            "errors": errors, "empty_hs_count": len(empty_codes), "coverage": rep}

# 실행
if RUN_PROBE_10DIGIT:
    probe_10digit()
    out = None
else:
    out = main()

# 결과 변수 편의 할당
if out:
    final_raw = out["final_raw"]
    monthly_trade_data = out["monthly"]
    quarterly_trade_data = out["quarterly"]
    monthly_long_data = out["monthly_long"]
    quarterly_long_data = out["quarterly_long"]
    error_list = out["errors"]
    coverage_report = out["coverage"]

    print("\n사용 가능한 변수들:")
    print("- final_raw")
    print("- monthly_trade_data")
    print("- quarterly_trade_data")
    print("- monthly_long_data")
    print("- quarterly_long_data")
    print("- error_list")
    print("- coverage_report")

# -----------------------------------------------------------------------------
# 시각화 함수(유지)
# -----------------------------------------------------------------------------
def plot_column_by_hscode(df, hs_code, col_name, start_date=None, end_date=None):
    import matplotlib.pyplot as plt

    if df is None or df.empty:
        print("DataFrame이 비어 있습니다.")
        return
    if "date" not in df.columns:
        print("'date' 컬럼이 없습니다. add_yoy_growth()를 먼저 실행하세요.")
        return

    hs_code = str(hs_code)
    target_df = df[df["root_hs_code"].astype(str) == hs_code].sort_values("date")

    if target_df.empty:
        print(f" root_hs_code {hs_code}에 해당하는 데이터가 없습니다.")
        return

    if col_name not in target_df.columns:
        print(f" '{col_name}' 컬럼이 DataFrame에 없습니다.")
        return

    if start_date:
        target_df = target_df[target_df["date"] >= pd.to_datetime(start_date)]
    if end_date:
        target_df = target_df[target_df["date"] <= pd.to_datetime(end_date)]

    if target_df.empty:
        print("지정한 날짜 범위에 데이터가 없습니다.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(target_df["date"], target_df[col_name], marker="o", label=col_name)

    last_x = target_df["date"].iloc[-1]
    last_y = target_df[col_name].iloc[-1]

    if "yoy" in col_name:
        plt.text(last_x, last_y, f"{last_y * 100:,.2f}%", fontsize=12, ha="left", va="bottom", color="red")
    else:
        plt.text(last_x, last_y, f"{last_y:,.0f}", fontsize=12, ha="left", va="bottom", color="red")

    plt.title(f"{col_name} 추이 (root_hs_code: {hs_code})")
    plt.xlabel("Date")
    plt.ylabel(col_name)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

[경로 설정] 자동 탐색 성공 | 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[경로 설정] DATA 폴더: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


2026-11-19 22:00:10,328 - INFO - 수집 목록 760개 (정적 562, DB 수요 추가 198)
2026-11-19 22:00:10,328 - INFO - DB 수요로 추가된 코드: ['1201', '151550', '151590', '1604', '1703', '190110', '1902', '200830', '2009', '210500', '220600', '220890', '250100', '261800', '2711', '280461', '2811', '2821', '282110', '290539', '291720', '292151', '292241', '292242', '292250', '293399', '294130', '300212', '300421', '300430', '300640', '3204', '3215', '3304', '3305', '330730', '340212', '350790', '360610', '370199', '370298', '381230', '3824', '382490', '391290', '391690', '3923', '392590', '4000', '400211']


[수요 코드] fd_driver_map_2d.code: 24개
[수요 코드] kr_company_hs_map.hs_code: 501개
[수요 코드] korea_company_region_hscode_map.hs_code: 3개


2026-11-19 22:00:11,683 - INFO - 커버리지 (기준월 2026-09 ): ok 561 | 끊김 164 | 처음 35
2026-11-19 22:00:11,686 - WARNING - 끊긴 코드 (최신월): 851780(2007-01), 852851(2017-07), 382490(2018-03), 854140(2022-03), 940130(2022-03), 741999(2022-03), 851770(2022-09), 940540(2022-10), 852580(2023-12), 732182(2025-07), 852872(2025-07), 852190(2025-07), 850819(2025-07), 732290(2025-07), 820740(2025-07), 710510(2025-07), 701349(2025-07), 540249(2025-07), 691490(2025-07), 847130(2025-07), 846229(2025-07), 560749(2025-07), 591110(2025-07), 844339(2025-07), 842410(2025-07), 842131(2025-07), 841939(2025-07), 370199(2025-07), 360610(2025-07), 950300(2025-07), 740990(2025-07), 902610(2025-07), 901819(2025-07), 901210(2025-07), 900691(2025-07), 350790(2025-07), 200830(2025-07), 903040(2025-07), 151590(2025-07), 282110(2025-07), 854710(2025-07), 293399(2025-07), 294130(2025-07), 300212(2025-07), 300640(2025-07), 854470(2025-07), 721730(2025-07), 151550(2025-07), 853340(2025-07), 290539(2025-07), 902750(2025-07), 8404(2

,code,name,sources,status,last_date,api_empty,note
96,300421,(DB 수요 코드),kr_company_hs_map(10->6),new,NaT,True,API 빈 응답 - HS 개정·단종 코드일 수 있음 (확인 필요)
97,300430,(DB 수요 코드),kr_company_hs_map(10->6),new,NaT,True,API 빈 응답 - HS 개정·단종 코드일 수 있음 (확인 필요)
121,340212,(DB 수요 코드),kr_company_hs_map(10->6),stale,2021-12-31,False,
137,381230,(DB 수요 코드),kr_company_hs_map(10->6),stale,2017-01-31,False,
143,382490,(DB 수요 코드),kr_company_hs_map(10->6),stale,2018-03-31,False,
176,391290,(DB 수요 코드),kr_company_hs_map(10->6),stale,2010-12-31,False,
177,391690,(DB 수요 코드),kr_company_hs_map(10->6),new,NaT,True,API 빈 응답 - HS 개정·단종 코드일 수 있음 (확인 필요)
190,3923,(DB 수요 코드),kr_company_hs_map,stale,2026-01-31,True,API 빈 응답 - HS 개정·단종 코드일 수 있음 (확인 필요)
196,392590,(DB 수요 코드),kr_company_hs_map,stale,2026-01-31,True,API 빈 응답 - HS 개정·단종 코드일 수 있음 (확인 필요)
199,4000,(DB 수요 코드),kr_company_hs_map,new,NaT,True,API 빈 응답 - HS 개정·단종 코드일 수 있음 (확인 필요)


2026-11-19 22:03:41,423 - INFO - 커버리지 리포트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\reports\trade_coverage\trade_coverage_20261119_2203.csv
2026-11-19 22:03:41,424 - INFO - 전체 파이프라인 완료



사용 가능한 변수들:
- final_raw
- monthly_trade_data
- quarterly_trade_data
- monthly_long_data
- quarterly_long_data
- error_list
- coverage_report


In [2]:
monthly_df = out['monthly']
monthly_df
monthly_df[monthly_df['root_hs_code'] == '854232'].tail(13)

,root_hs_code,new_year,new_quarter,new_month,balPayments,expDlr,impDlr,expWgt,impWgt,region,date,expDlr_yoy,impDlr_yoy


In [3]:
# date 인덱스 생성
monthly_df['date'] = pd.to_datetime(
    monthly_df['new_year'].astype(str) + '-' + monthly_df['new_month'].astype(str).str.zfill(2)
)

# pivot table 생성
pivot_df = monthly_df.pivot_table(
    index='date',
    columns='root_hs_code',
    values='expDlr',
    aggfunc='sum'
)

pivot_df.index = pd.DatetimeIndex(pivot_df.index).to_period('M')

print(pivot_df)

root_hs_code       0207    030341     030342      030343     030344  \
date                                                                  
2005-10             NaN       NaN        NaN         NaN        NaN   
2005-11             NaN       NaN        NaN         NaN        NaN   
2005-12             NaN       NaN        NaN         NaN        NaN   
2006-01             NaN       NaN        NaN         NaN        NaN   
2006-02             NaN       NaN        NaN         NaN        NaN   
...                 ...       ...        ...         ...        ...   
2026-06       3963221.0  498148.0  5604700.0  13167232.0  2907825.0   
2026-07       5851999.0  416523.0  7337526.0  11462398.0  3413307.0   
2026-08       4565948.0   96979.0  1302317.0   2116431.0   976602.0   
2026-09       6253090.0  737076.0  2660410.0   4197589.0  4346559.0   
2026-10       8036259.0  109375.0  5368310.0  10236263.0  1106116.0   

root_hs_code      030354      030389      030487     030633     1201  ...  \

In [4]:
# pivot_df.to_csv(r'C:\Users\82108\OneDrive\INVESTMENT\Data_Asnal\Export_data\export_202602.csv')

In [10]:
pivot_df[['854232']].tail(10)

KeyError: "None of [Index(['854232'], dtype='object', name='root_hs_code')] are in the [columns]"

In [12]:
pivot_df.tail(12)

root_hs_code,0207,030341,030342,030343,030344,030354,030389,030487,030633,1201,...,930690,940130,940199,940340,9405,940540,950300,961610,961900,970191
date,,,,,,,,,,,,,,,,,,,,,
2025-11,7399151.0,229904.0,4366349.0,19342111.0,1474513.0,18508698.0,7600561.0,22436329.0,9299194.0,6892.0,...,64523371.0,NaN,61732717.0,547684.0,10556729.0,NaN,7793926.0,298937.0,4385115.0,8310730.0
2025-12,7057386.0,692036.0,5348162.0,14251838.0,423604.0,54414965.0,9042035.0,22134387.0,13243820.0,35447.0,...,151286954.0,NaN,73329621.0,379747.0,12907164.0,NaN,9888645.0,312966.0,4871363.0,18758405.0
2026-01,7762314.0,349946.0,1140427.0,1699040.0,495588.0,32386717.0,11555723.0,25513365.0,10648201.0,17403.0,...,143640028.0,NaN,62530740.0,164272.0,10879071.0,NaN,9400025.0,337268.0,5194911.0,19821860.0
2026-02,5053709.0,741631.0,2453410.0,4811896.0,3906275.0,40680793.0,8249015.0,19867057.0,11736717.0,9799.0,...,105063807.0,NaN,52761223.0,364524.0,10072623.0,NaN,7092355.0,238576.0,3442530.0,7408203.0
2026-03,7657507.0,726726.0,3435609.0,1499918.0,3404168.0,28852662.0,7404192.0,22158262.0,4148360.0,6119.0,...,247510129.0,NaN,67325396.0,1001290.0,10855328.0,NaN,8365210.0,505069.0,4680270.0,25901325.0
2026-04,5243114.0,726074.0,5955960.0,21608352.0,3695398.0,26938583.0,9058483.0,27919571.0,8590802.0,15509.0,...,780334.0,NaN,71036682.0,956319.0,11316731.0,NaN,7876255.0,611519.0,4357759.0,8447198.0
2026-05,3929517.0,680576.0,4214018.0,16649865.0,4555163.0,23767546.0,10226546.0,32011194.0,8004630.0,11793.0,...,51919871.0,NaN,56673533.0,224902.0,10524631.0,NaN,6946383.0,677974.0,3227930.0,7016953.0
2026-06,3963221.0,498148.0,5604700.0,13167232.0,2907825.0,27340340.0,7854003.0,32818200.0,6915413.0,47365.0,...,76703667.0,NaN,58683248.0,522348.0,12575485.0,NaN,8057563.0,506012.0,4534225.0,9067257.0
2026-07,5851999.0,416523.0,7337526.0,11462398.0,3413307.0,18212100.0,10892117.0,26623797.0,3927527.0,10935.0,...,51749700.0,NaN,73751433.0,559276.0,10952463.0,NaN,9315924.0,480215.0,4654024.0,10583072.0
